## Imports

In [46]:
import os
import glob
from pathlib import Path
import yaml

from PIL import Image
import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader

import yolov5
from yolov5.models.yolo import Model
from yolov5.utils.general import check_yaml, yaml_load, intersect_dicts
from yolov5.utils.loss import ComputeLoss

## Util functions

In [47]:
def collate_fn(batch):
    imgs, targets = list(zip(*batch))
    targets = list(targets)
    for i, boxes in enumerate(targets):
        if boxes.numel():
            targets[i] = torch.cat((torch.full((boxes.shape[0], 1), i), boxes), dim=1)
        else:
            targets[i] = torch.zeros((0, 6))
    return torch.stack(imgs), torch.cat(targets, dim=0)

## Data loader

In [48]:
class YoloDetectionDataset(Dataset):

    image_dir_name = 'images'
    label_dir_name = 'labels'

    def __init__(self, data_dir, img_size=640, transforms=None):
        self.images_paths = sorted(glob.glob(os.path.join(data_dir, self.image_dir_name, '*.jpg')))
        self.labels_dir = os.path.join(data_dir, self.label_dir_name)
        self.img_size = img_size
        self.transforms = transforms

    def __len__(self):
        return len(self.images_paths)

    def __getitem__(self, idx):
        img_path = self.images_paths[idx]
        label_path = os.path.join(
            self.labels_dir,
            os.path.basename(img_path).replace('.jpg', '.txt')
        )

        # Load image
        img = Image.open(img_path).convert('RGB')
        img = np.array(img)

        # Load labels
        if os.path.exists(label_path):
            labels = np.loadtxt(label_path).reshape(-1, 5)
        else:
            labels = np.zeros((0, 5))

        if self.transforms:
            transformed = self.transforms(image=img, bboxes=labels[:, 1:], class_labels=labels[:, 0])
            img = transformed['image']
            boxes = transformed['bboxes']
            classes = transformed['class_labels']
            labels = np.column_stack([classes, boxes])

        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        labels = torch.tensor(labels, dtype=torch.float32)
        return img, labels

In [49]:
train_dataset = YoloDetectionDataset('../dataset/train')
train_loader = DataLoader(
    train_dataset, 
    batch_size=8, 
    shuffle=True, 
    num_workers=0,
    collate_fn=collate_fn)

## Load YOLOv5 nano model weights

In [ ]:
# Load yolov5n.yaml from package
yolov5_dir = Path(yolov5.__file__).resolve().parent
model_cfg = yolov5_dir / 'models' / 'yolov5n.yaml'

In [51]:
# Build model
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu')
data_yaml = check_yaml('../dataset/data.yaml')
data_dict = yaml_load(data_yaml)
nc = data_dict.get('nc', 1)

model = Model(str(model_cfg), ch=3, nc=nc).to(device)
model.float()  # ensure float32

Overriding model.yaml nc=80 with nc=6

                 from  n    params  module                                  arguments                     
  0                -1  1      1760  yolov5.models.common.Conv               [3, 16, 6, 2, 2]              
  1                -1  1      4672  yolov5.models.common.Conv               [16, 32, 3, 2]                
  2                -1  1      4800  yolov5.models.common.C3                 [32, 32, 1]                   
  3                -1  1     18560  yolov5.models.common.Conv               [32, 64, 3, 2]                
  4                -1  2     29184  yolov5.models.common.C3                 [64, 64, 2]                   
  5                -1  1     73984  yolov5.models.common.Conv               [64, 128, 3, 2]               
  6                -1  3    156928  yolov5.models.common.C3                 [128, 128, 3]                 
  7                -1  1    295424  yolov5.models.common.Conv               [128, 256, 3, 2]             

DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(6, 6), stride=(2, 2), padding=(2, 2), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3(
      (cv1): Conv(
        (conv): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
     

In [52]:
# Load pretrained weights on CPU
weights_url = "https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5n.pt"
checkpoint = torch.hub.load_state_dict_from_url(weights_url, map_location='cpu')

In [ ]:
# We output 6 classes, not 80 like the original model
# So we keep only layers whose shapes match
checkpoint_dict = checkpoint['model'].float().state_dict()
model_dict = model.state_dict()
compatible_weights = intersect_dicts(checkpoint_dict, model_dict, exclude=['anchors'])
model_dict.update(compatible_weights)
model.load_state_dict(model_dict)

<All keys matched successfully>

In [ ]:
# Move model to GPU (if available)
model.to(device)

DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(6, 6), stride=(2, 2), padding=(2, 2), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3(
      (cv1): Conv(
        (conv): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
     

In [ ]:
with open('../yolov5/hyp.scratch-low.yaml') as f:
    hyp = yaml.safe_load(f)         # load model hyper params dict

model.hyp = hyp                     # hyper params
model.nc = nc                       # number of classes
model.names = data_dict['names']    # for completeness

criterion = ComputeLoss(model)

## Optimizer and Scheduler

In [56]:
optimizer = torch.optim.SGD(
    model.parameters(), 
    lr=0.01, 
    momentum=0.937, 
    weight_decay=5e-4
)

lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

## Training loop

In [57]:
epochs = 50
model.train()

for epoch in range(epochs):
    total_loss = 0
    for imgs, targets in train_loader:
        imgs = imgs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        preds = model(imgs)  # list of tensors at different scales
        loss, loss_items = criterion(preds, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    lr_scheduler.step()
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

    # Optional validation step (e.g., compute mAP)

torch.save(model.state_dict(), 'yolov5n_custom.pt')

Epoch 1/50 - Loss: 6.4510
Epoch 2/50 - Loss: 5.4788
Epoch 3/50 - Loss: 5.4477
Epoch 4/50 - Loss: 5.0659
Epoch 5/50 - Loss: 4.7124
Epoch 6/50 - Loss: 4.7821
Epoch 7/50 - Loss: 4.8435
Epoch 8/50 - Loss: 4.5914
Epoch 9/50 - Loss: 4.8182
Epoch 10/50 - Loss: 4.5946
Epoch 11/50 - Loss: 4.1841
Epoch 12/50 - Loss: 4.2157
Epoch 13/50 - Loss: 4.0422
Epoch 14/50 - Loss: 4.1000
Epoch 15/50 - Loss: 3.9791
Epoch 16/50 - Loss: 4.1211
Epoch 17/50 - Loss: 4.0705
Epoch 18/50 - Loss: 3.7376
Epoch 19/50 - Loss: 3.8286
Epoch 20/50 - Loss: 3.6782
Epoch 21/50 - Loss: 3.4651
Epoch 22/50 - Loss: 3.2229
Epoch 23/50 - Loss: 3.2462
Epoch 24/50 - Loss: 3.5401
Epoch 25/50 - Loss: 3.3791
Epoch 26/50 - Loss: 3.1496
Epoch 27/50 - Loss: 2.8646
Epoch 28/50 - Loss: 2.9618
Epoch 29/50 - Loss: 2.8494
Epoch 30/50 - Loss: 2.7162
Epoch 31/50 - Loss: 2.7798
Epoch 32/50 - Loss: 2.7207
Epoch 33/50 - Loss: 2.5218
Epoch 34/50 - Loss: 2.4808
Epoch 35/50 - Loss: 2.3099
Epoch 36/50 - Loss: 2.2893
Epoch 37/50 - Loss: 2.2209
Epoch 38/5